In [29]:
import pandas as pd
import numpy as np

In [30]:
df = pd.read_csv('house_price_prediction.csv')

In [31]:
df.shape

(4600, 18)

In [32]:
df = df.drop(["statezip", "country", "street"], axis=1)

In [33]:
df.head()

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,city
0,02/05/2014 0:00,313000.0,3,1.50,1340,7912,1.5,0,0,3,1340,0,1955,2005,Shoreline
1,02/05/2014 0:00,2384000.0,5,2.50,3650,9050,2.0,0,4,5,3370,280,1921,0,Seattle
2,02/05/2014 0:00,342000.0,3,2.00,1930,11947,1.0,0,0,4,1930,0,1966,0,Kent
3,02/05/2014 0:00,420000.0,3,2.25,2000,8030,1.0,0,0,4,1000,1000,1963,0,Bellevue
4,02/05/2014 0:00,550000.0,4,2.50,1940,10500,1.0,0,0,4,1140,800,1976,1992,Redmond


In [34]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder

In [35]:
# Suppose city column is a Series
city_col = df['city']

# 1. Reshape to 2D
city_reshaped = city_col.values.reshape(-1,1)  # shape (n_samples, 1)

# 2. Initialize encoder
ohe = OneHotEncoder(sparse_output=False, drop='first')  # updated argument

# 3. Fit and transform
city_encoded = ohe.fit_transform(city_reshaped)

# 4. Convert to DataFrame with column names
city_df = pd.DataFrame(city_encoded, columns=ohe.get_feature_names_out(['city']))

# 5. Drop original column and concat
df = df.drop('city', axis=1)
df = pd.concat([df, city_df], axis=1)

df['date'] = pd.to_datetime(df['date'], format="%d/%m/%Y %H:%M")
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df["age"] = df["year"] - df["yr_built"]
df['renovated'] = df['yr_renovated'].apply(lambda x: 0 if x == 0 else 1)
df = df.drop(["date", "yr_built", "sqft_above", "sqft_basement", "sqft_lot", "yr_renovated", "bedrooms","year"], axis=1)
df = df[df['price'] > 0]
df["price"] = np.log(df['price'])


In [36]:
df.shape

(4551, 54)

In [37]:
X = df.drop('price', axis=1)
y = df['price']

In [38]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [39]:
model = LinearRegression()
model.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [40]:
y_pred = model.predict(X_test)

In [41]:
coeff_dr = pd.DataFrame(model.coef_, X.columns, columns=["Coefficient"])
print(coeff_dr)

                           Coefficient
bathrooms                 7.667739e-02
sqft_living               2.791223e-04
floors                    1.058344e-01
waterfront                2.709590e-01
view                      7.306891e-02
condition                 6.878242e-02
city_Auburn               7.117111e-02
city_Beaux Arts Village  -1.713560e-14
city_Bellevue             8.249608e-01
city_Black Diamond        4.201747e-01
city_Bothell              5.026401e-01
city_Burien               2.112386e-01
city_Carnation            2.858606e-01
city_Clyde Hill           1.298637e+00
city_Covington            1.071275e-03
city_Des Moines           1.166603e-01
city_Duvall               3.420067e-01
city_Enumclaw             8.275074e-02
city_Fall City            5.541773e-01
city_Federal Way          5.832643e-02
city_Inglewood-Finn Hill  9.228729e-16
city_Issaquah             6.161385e-01
city_Kenmore              5.065479e-01
city_Kent                 1.481264e-01
city_Kirkland            

In [42]:
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Mean Squared Error = ", mse)
print("R2 Scor = ", r2)

Mean Squared Error =  0.10022694688283248
R2 Scor =  0.6578411199787859
